# 📊 Project Cartensz: Evaluasi & Metrik (M3)

> **Sistem Pakar Deteksi Narasi Ancaman PT Gemilang Satria Perkasa**  
> Catatan ini menayangkan rapor evaluasi kuantitatif dari model lokal SetFit.

---

### 🎯 Syarat Kelulusan Tayang:
1. **Skor `Weighted F1` minimal 0.70**
2. **Presisi `TINGGI` minimal 0.75** (untuk menekan alarm palsu agar analis lapangan tidak gampang panik/lelah)

In [ ]:
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# tarik rapor kelulusan dari mesin latih (evaluate.py)
with open('reports/evaluation_report.json', 'r', encoding='utf-8') as f:
    report = json.load(f)

print(f"Model: {report['model']} ({report['backbone']})")
print(f"Test Set Size: {report['test_size']}")

## 1. Inspeksi Syarat Tayang (KPI) 🚦

In [ ]:
pc = report['passing_criteria']
print("=" * 40)
print("CEK PASSING CRITERIA")
print("=" * 40)
f1_status = '✅ LULUS' if pc['weighted_f1_pass'] else '❌ GAGAL'
tp_status = '✅ LULUS' if pc['tinggi_precision_pass'] else '❌ GAGAL'
print(f"Weighted F1 ≥ {pc['weighted_f1_min']}:     {f1_status} ({report['overall_metrics']['weighted_f1']:.4f})")
print(f"TINGGI Prec ≥ {pc['tinggi_precision_min']}: {tp_status} ({report['per_class_metrics']['TINGGI']['precision']:.4f})")
print("=" * 40)

## 2. Pembedahan Angka Tiap Kelas 📈

In [ ]:
per_class_df = pd.DataFrame(report['per_class_metrics']).T
display(per_class_df.style.background_gradient(cmap='Greens', subset=['precision', 'recall', 'f1_score']))

## 3. Matriks Galat Kebingungan (Confusion Matrix) 🔲

In [ ]:
labels = report['confusion_matrix_labels']
cm = report['confusion_matrix']

cm_df = pd.DataFrame(cm, index=[f"Fakta {l}" for l in labels], columns=[f"Tebakan {l}" for l in labels])

plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='g', cmap='Blues', cbar=False)
plt.title('Matriks Klasifikasi Galat - Model SetFit Lokal')
plt.yticks(rotation=0)
plt.show()

## 4. Pelacakan Pola Kecacatan 🔍
Membongkar dimana model ini rentan kepeleset. Mengingat sifat tugas yang mengutamakan keawasan, alarm palsu tinggi (AMAN tebak WASPADA/TINGGI) bisa dimaklumi ketimbang kecolongan.

In [ ]:
print(f"Rasio Meleset: {report['error_rate']:.2%}")
print("\nPola Keliru Terbanyak:")
patterns = report['error_patterns']
for p, count in sorted(patterns.items(), key=lambda x: -x[1]):
    print(f"  - {p}: {count}")

## 5. Rapor Klasifikasi Lengkap Scikit-Learn

In [ ]:
print(report['classification_report'])